# 🔷 Nhận diện hình học — Huấn luyện trên Google Colab

Notebook tự động: **sinh dataset → train model CNN → tải model về máy**.
Không cần upload gì — dataset 12.000 ảnh được sinh ngay trên Colab.

## Cách dùng
1. **Bật GPU**: menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → `Save`.
2. `Runtime` → `Run all` (hoặc chạy lần lượt từng cell từ trên xuống).
3. Cell cuối tự tải `shape_model.keras` + `class_names.json` về máy bạn.
4. Chép 2 file đó vào thư mục `ml/models/` của project.

⏱️ Thời gian: khoảng 10–20 phút tùy GPU.


## Bước 0 — Kiểm tra GPU

In [ ]:
# Kiểm tra GPU đã bật chưa
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU san sang — train se nhanh.")
else:
    print("CHUA BAT GPU. Vao: Runtime > Change runtime type > T4 GPU.")
    print("(Khong bat van chay duoc nhung rat cham.)")


## Bước 1 — Sinh dataset (6 hình × 2000 ảnh)

In [ ]:
# -*- coding: utf-8 -*-
"""
Sinh dữ liệu ảnh hình học (6 loại hình, 2000 ảnh/loại)
- Tam giác, Vuông, Chữ nhật, Tròn, Ngũ giác, Lục giác
- Augmentation: rotation, brightness, contrast, noise, position random
- Size: 128×128 RGB
"""

import os
import cv2
import numpy as np
import random
from pathlib import Path

# ============================================================================
# CẤU HÌNH
# ============================================================================
IMG_SIZE = 128
IMAGES_PER_SHAPE = 2000
SCRIPT_DIR = Path("/content")
OUTPUT_DIR = SCRIPT_DIR / "images"
SHAPES = ["tam_giac", "vuong", "chu_nhat", "tron", "ngu_giac", "luc_giac"]

# ============================================================================
# TỰA HÀM VẼ HÌNH
# ============================================================================

def draw_triangle(img, center, size, angle):
    """Vẽ tam giác."""
    points = np.array([
        [center[0], center[1] - size],
        [center[0] - size, center[1] + size * 0.866],
        [center[0] + size, center[1] + size * 0.866]
    ], dtype=np.int32)

    # Rotate
    angle_rad = np.radians(angle)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    points_rot = []
    for p in points:
        dx, dy = p[0] - center[0], p[1] - center[1]
        new_x = int(center[0] + dx * cos_a - dy * sin_a)
        new_y = int(center[1] + dx * sin_a + dy * cos_a)
        points_rot.append([new_x, new_y])
    points_rot = np.array(points_rot, dtype=np.int32)

    cv2.drawContours(img, [points_rot], 0, (0, 0, 0), 2)
    return img

def draw_square(img, center, size, angle):
    """Vẽ hình vuông."""
    half_size = int(size / 1.414)
    points = np.array([
        [center[0] - half_size, center[1] - half_size],
        [center[0] + half_size, center[1] - half_size],
        [center[0] + half_size, center[1] + half_size],
        [center[0] - half_size, center[1] + half_size]
    ], dtype=np.int32)

    angle_rad = np.radians(angle)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    points_rot = []
    for p in points:
        dx, dy = p[0] - center[0], p[1] - center[1]
        new_x = int(center[0] + dx * cos_a - dy * sin_a)
        new_y = int(center[1] + dx * sin_a + dy * cos_a)
        points_rot.append([new_x, new_y])
    points_rot = np.array(points_rot, dtype=np.int32)

    cv2.drawContours(img, [points_rot], 0, (0, 0, 0), 2)
    return img

def draw_rectangle(img, center, size, angle):
    """Vẽ hình chữ nhật."""
    width = int(size * 1.5)
    height = int(size * 0.8)
    points = np.array([
        [center[0] - width // 2, center[1] - height // 2],
        [center[0] + width // 2, center[1] - height // 2],
        [center[0] + width // 2, center[1] + height // 2],
        [center[0] - width // 2, center[1] + height // 2]
    ], dtype=np.int32)

    angle_rad = np.radians(angle)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    points_rot = []
    for p in points:
        dx, dy = p[0] - center[0], p[1] - center[1]
        new_x = int(center[0] + dx * cos_a - dy * sin_a)
        new_y = int(center[1] + dx * sin_a + dy * cos_a)
        points_rot.append([new_x, new_y])
    points_rot = np.array(points_rot, dtype=np.int32)

    cv2.drawContours(img, [points_rot], 0, (0, 0, 0), 2)
    return img

def draw_circle(img, center, size, angle=0):
    """Vẽ hình tròn."""
    cv2.circle(img, center, size, (0, 0, 0), 2)
    return img

def draw_pentagon(img, center, size, angle):
    """Vẽ ngũ giác."""
    points = []
    for i in range(5):
        theta = np.radians(angle) + (i * 72) * np.pi / 180
        x = int(center[0] + size * np.cos(theta))
        y = int(center[1] + size * np.sin(theta))
        points.append([x, y])
    points = np.array(points, dtype=np.int32)
    cv2.drawContours(img, [points], 0, (0, 0, 0), 2)
    return img

def draw_hexagon(img, center, size, angle):
    """Vẽ lục giác."""
    points = []
    for i in range(6):
        theta = np.radians(angle) + (i * 60) * np.pi / 180
        x = int(center[0] + size * np.cos(theta))
        y = int(center[1] + size * np.sin(theta))
        points.append([x, y])
    points = np.array(points, dtype=np.int32)
    cv2.drawContours(img, [points], 0, (0, 0, 0), 2)
    return img

# ============================================================================
# AUGMENTATION
# ============================================================================

def create_background():
    """Tạo background gradient: trắng/xám sáng → xám tối."""
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    # Gradient từ trên xuống (trắng → xám tối)
    for y in range(IMG_SIZE):
        gray_val = int(255 * (1 - y / IMG_SIZE * 0.7))  # 255 → 75
        img[y, :] = [gray_val, gray_val, gray_val]

    return img

def apply_brightness_contrast(img, brightness=0, contrast=1):
    """Áp dụng brightness & contrast."""
    img = img.astype(np.float32)
    img = img * contrast + brightness
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def apply_gaussian_noise(img, sigma=10):
    """Thêm Gaussian noise."""
    noise = np.random.normal(0, sigma, img.shape)
    img = img.astype(np.float32) + noise
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def augment_image(img):
    """Áp dụng tất cả augmentation."""
    # Brightness: ±30%
    brightness = random.uniform(-30, 30)
    # Contrast: ±20%
    contrast = random.uniform(0.8, 1.2)
    img = apply_brightness_contrast(img, brightness, contrast)

    # Gaussian noise
    if random.random() > 0.5:
        img = apply_gaussian_noise(img, sigma=random.uniform(5, 15))

    # Blur nhẹ
    if random.random() > 0.7:
        img = cv2.GaussianBlur(img, (3, 3), 0)

    return img

# ============================================================================
# SINH DỮ LIỆU
# ============================================================================

def generate_dataset():
    """Sinh dữ liệu 6 loại hình."""
    draw_funcs = {
        "tam_giac": draw_triangle,
        "vuong": draw_square,
        "chu_nhat": draw_rectangle,
        "tron": draw_circle,
        "ngu_giac": draw_pentagon,
        "luc_giac": draw_hexagon
    }

    for shape_name in SHAPES:
        shape_dir = OUTPUT_DIR / shape_name
        shape_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n📊 Sinh dữ liệu: {shape_name}")

        for i in range(IMAGES_PER_SHAPE):
            # Tạo background
            img = create_background()

            # Size: random (25-45 pixel)
            size = random.randint(25, 45)

            # Vị trí hình: random nhưng đảm bảo hình không bị cắt mép
            margin = size + 6
            center_x = random.randint(margin, IMG_SIZE - margin)
            center_y = random.randint(margin, IMG_SIZE - margin)
            center = (center_x, center_y)

            # Góc: 0-360 độ (tối ưu hóa)
            angle = random.uniform(0, 360)

            # Vẽ hình
            draw_func = draw_funcs[shape_name]
            img = draw_func(img, center, size, angle)

            # Augmentation
            img = augment_image(img)

            # Lưu PNG
            filename = f"{shape_name}_{i:04d}.png"
            filepath = shape_dir / filename
            cv2.imwrite(str(filepath), img)

            if (i + 1) % 200 == 0:
                print(f"  ✓ Đã sinh {i + 1}/{IMAGES_PER_SHAPE} ảnh")

        print(f"✅ Hoàn thành: {shape_name}")

    print(f"\n🎉 Sinh dữ liệu thành công! Vị trí: {OUTPUT_DIR}")
    print(f"   Tổng: {len(SHAPES)} hình × {IMAGES_PER_SHAPE} ảnh = {len(SHAPES) * IMAGES_PER_SHAPE} ảnh")

if __name__ == "__main__":
    generate_dataset()


## Bước 2 — Huấn luyện model CNN

In [ ]:
# -*- coding: utf-8 -*-
"""
Train model CNN với Residual Blocks cho nhận diện 6 loại hình
- Kiến trúc: Residual CNN (Option 1)
- Input: 128×128×3 (RGB)
- Output: 6 classes
"""

import os

# Tắt oneDNN/MKL của TensorFlow để tránh lỗi "could not create a memory object"
# (mkl_conv_grad_filter_ops.cc) khi train CNN trên CPU Windows.
# PHẢI đặt TRƯỚC khi import tensorflow.
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import matplotlib.pyplot as plt

# ============================================================================
# CẤU HÌNH
# ============================================================================
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 100
SCRIPT_DIR = Path("/content")
DATA_DIR = SCRIPT_DIR / "images"
MODEL_DIR = SCRIPT_DIR / "models"
LOG_DIR = SCRIPT_DIR / "logs"

SHAPES = ["tam_giac", "vuong", "chu_nhat", "tron", "ngu_giac", "luc_giac"]
NUM_CLASSES = len(SHAPES)
RANDOM_SEED = 42

# Reproducibility: kết quả train lặp lại được
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ============================================================================
# LOAD DỮ LIỆU
# ============================================================================

def load_dataset():
    """Load ảnh từ folder."""
    images = []
    labels = []

    for label_idx, shape_name in enumerate(SHAPES):
        shape_dir = DATA_DIR / shape_name
        if not shape_dir.exists():
            print(f"⚠️  Folder không tồn tại: {shape_dir}")
            continue

        files = sorted(list(shape_dir.glob("*.png")))
        print(f"📖 Load {shape_name}: {len(files)} ảnh")

        for img_path in files:
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR → RGB
            img = img.astype(np.float32) / 255.0  # Normalize [0, 1]
            images.append(img)
            labels.append(label_idx)

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels)

    # One-hot encoding
    labels = keras.utils.to_categorical(labels, NUM_CLASSES)

    print(f"\n✅ Load dữ liệu hoàn tất:")
    print(f"   Images shape: {images.shape}")
    print(f"   Labels shape: {labels.shape}")

    return images, labels

# ============================================================================
# BUILD MODEL - RESIDUAL CNN (OPTION 1)
# ============================================================================

def residual_block(x, filters, kernel_size=3, activation="relu"):
    """Residual Block: Conv + BN + ReLU + Conv + BN + ReLU + Skip connection."""
    x_input = x

    x = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)

    x = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)

    # Skip connection
    if x_input.shape[-1] == filters:
        x = layers.Add()([x, x_input])
    else:
        x_input = layers.Conv2D(filters, 1, padding="same")(x_input)
        x = layers.Add()([x, x_input])

    x = layers.Activation(activation)(x)

    return x

def build_model():
    """Build Residual CNN model (augmentation tích hợp trong graph)."""
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Augmentation nằm TRONG model: tự động ON khi fit(), OFF khi predict/evaluate
    x = create_data_augmentation()(inputs)

    # BLOCK 1: 32 filters
    x = layers.Conv2D(32, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 32)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D(2)(x)  # → 64×64

    # BLOCK 2: 64 filters
    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D(2)(x)  # → 32×32

    # BLOCK 3: 128 filters
    x = layers.Conv2D(128, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 128)
    x = residual_block(x, 128)
    x = layers.MaxPooling2D(2)(x)  # → 16×16

    # BLOCK 4: 256 filters
    x = layers.Conv2D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 256)
    x = residual_block(x, 256)
    x = layers.MaxPooling2D(2)(x)  # → 8×8

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dense layers
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    # Output
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model

# ============================================================================
# DATA AUGMENTATION
# ============================================================================

def create_data_augmentation():
    """Augmentation pipeline — random lại mỗi batch/epoch khi training."""
    return keras.Sequential([
        layers.RandomRotation(0.2, seed=RANDOM_SEED),
        layers.RandomBrightness(0.2, value_range=(0.0, 1.0), seed=RANDOM_SEED),
        layers.RandomContrast(0.2, seed=RANDOM_SEED),
        layers.RandomFlip("horizontal_and_vertical", seed=RANDOM_SEED),
        layers.RandomZoom(0.15, seed=RANDOM_SEED),
    ], name="data_augmentation")

# ============================================================================
# TRAIN MODEL
# ============================================================================

def train_model():
    """Train model."""
    # Load dữ liệu
    images, labels = load_dataset()

    # Train/Val/Test split
    x_train, x_temp, y_train, y_temp = train_test_split(
        images, labels, test_size=0.30, random_state=42
    )
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp, y_temp, test_size=0.50, random_state=42
    )

    print(f"\n📊 Data split:")
    print(f"   Train: {x_train.shape[0]} ảnh")
    print(f"   Val:   {x_val.shape[0]} ảnh")
    print(f"   Test:  {x_test.shape[0]} ảnh")

    # Build model
    print(f"\n🏗️  Build Residual CNN model...")
    model = build_model()
    model.summary()

    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Callbacks
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    )

    model_checkpoint = keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / "shape_model_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )

    csv_logger = keras.callbacks.CSVLogger(
        str(LOG_DIR / "training_history.csv"),
        append=False
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )

    # Train (augmentation đã nằm trong model nên truyền thẳng x_train)
    print(f"\n🚀 Training model...")
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=[early_stopping, model_checkpoint, csv_logger, reduce_lr],
        verbose=1
    )

    # Evaluate on test set
    print(f"\n📈 Evaluate on test set...")
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"   Test Loss:     {test_loss:.4f}")
    print(f"   Test Accuracy: {test_acc:.4f}")

    # Báo cáo per-class (precision/recall/f1) + confusion matrix
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    y_true = np.argmax(y_test, axis=1)
    print("\n📋 Classification report (test set):")
    print(classification_report(y_true, y_pred, target_names=SHAPES, digits=4))
    plot_confusion_matrix(y_true, y_pred)

    # Save final model
    final_model_path = MODEL_DIR / "shape_model.keras"
    model.save(str(final_model_path))
    print(f"\n✅ Model lưu tại: {final_model_path}")

    # Save class names
    class_names_path = MODEL_DIR / "class_names.json"
    with open(str(class_names_path), "w", encoding="utf-8") as f:
        json.dump(SHAPES, f, ensure_ascii=False, indent=2)
    print(f"✅ Class names lưu tại: {class_names_path}")

    # Plot training history
    plot_history(history)

def plot_history(history):
    """Vẽ biểu đồ training history."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Accuracy
    axes[0].plot(history.history["accuracy"], label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Val")
    axes[0].set_title("Model Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    axes[1].plot(history.history["loss"], label="Train")
    axes[1].plot(history.history["val_loss"], label="Val")
    axes[1].set_title("Model Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    # Save figure
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(LOG_DIR / "training_history.png"), dpi=100, bbox_inches="tight")
    print(f"✅ Training history plot lưu tại: {LOG_DIR / 'training_history.png'}")

    plt.show()


def plot_confusion_matrix(y_true, y_pred):
    """Vẽ confusion matrix trên test set."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(SHAPES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(SHAPES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Test set)")

    thresh = cm.max() / 2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

    fig.colorbar(im)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    out_path = LOG_DIR / "confusion_matrix.png"
    plt.savefig(str(out_path), dpi=100, bbox_inches="tight")
    print(f"✅ Confusion matrix lưu tại: {out_path}")
    plt.close(fig)


if __name__ == "__main__":
    train_model()


## Bước 3 — Tải model về máy

In [ ]:
# Tải các file kết quả về máy (KHÔNG tải dataset ảnh)
import os
from google.colab import files

outputs = [
    "/content/models/shape_model.keras",        # model cuối cùng — dùng cho app
    "/content/models/shape_model_best.keras",   # model tốt nhất theo val_accuracy
    "/content/models/class_names.json",         # tên 6 lớp hình
    "/content/logs/training_history.csv",       # số liệu accuracy/loss từng epoch
    "/content/logs/training_history.png",       # biểu đồ accuracy & loss
    "/content/logs/confusion_matrix.png",       # ma trận nhầm lẫn (test set)
]

for path in outputs:
    if os.path.exists(path):
        print("Tai ve:", path)
        files.download(path)
    else:
        print("Bo qua (khong tim thay):", path)

print()
print("Xong! Sap xep file vao project:")
print("  *.keras + class_names.json                 -> ml/models/")
print("  training_history.* + confusion_matrix.png  -> ml/logs/")
